In [733]:
import requests
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.by import By
import pandas as pd
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.common.keys import Keys
import time

In [735]:
categories = ["CB", "C0", "C1", "C2"]
category_dict = {"CB" : [" CВ", " CB", " СВ", " СB"], "C0" : [" CO", " C0", " С0", " СО", " CО", " СO"], "C1" : [" C1", " С1"], "C2" :[" C2", " С2"]}

In [737]:
urls = {"perek" : "https://www.perekrestok.ru/cat/search?search=яйца", "pyaterka" : "https://5ka.ru/search/?value=яйца"}
goods_classes = {"perek" : "product-card__content", "pyaterka" : "chakra-stack.KnkuqE3h-.HmedBAdm-.css-8g8ihq"}
shops_available = ['perek', 'pyaterka']

In [739]:
def get_category(name : str):
    for category in category_dict:
        for spelling in category_dict[category]:
            if spelling in name:
                return category
    return None

In [741]:
def get_goods(shop : str):
    driver = webdriver.Chrome()
    driver.get(urls[shop])
    time.sleep(5)
    actions = ActionChains(driver)
    actions.send_keys(Keys.END).perform()
    time.sleep(5)
    actions.send_keys(Keys.END).perform()
    goods = driver.find_elements(By.CLASS_NAME, goods_classes[shop])
    return goods

In [743]:
def get_name(good : str, shop : str):
    if shop == "pyaterka":
        return good.find_element(By.CLASS_NAME, "chakra-text.SdLEFc2B-.css-1jdqp4k").text
    if shop == "perek":
        return good.find_element(By.CLASS_NAME, "product-card__title").text
        

In [745]:
def get_price(good : str, shop : str):
    if shop == "pyaterka":
        price_rub = int(good.find_element(By.CLASS_NAME, "chakra-text.DUXYWqnZ-.css-6uvdux").text)
        price_kop = int(good.find_element(By.CLASS_NAME, "chakra-text.j2bifgeA-.css-6uvdux").text)
        price = price_rub + price_kop / 100
        
    if shop == "perek":
        price_raw = good.find_element(By.CLASS_NAME, "price-new")
        price = float(price_raw.text.split('\n')[1][:-2].replace(',', '.'))
    return price

In [747]:
def get_shop_egg_prices(shop : str):
    assert shop in shops_available, f"Магазин '{shop}' не поддерживается в данный момент. Пожалуйста, выберите магазин из списка {', '.join(map(str, shops_availabe))}"
    names = []
    prices = []
    eggs_data_dict = {'Название': [], 'Категория': [], 'Цена': []}
    
    goods = get_goods(shop)
    for good in goods:
        names.append(get_name(good, shop))
        prices.append(get_price(good, shop))
    raw_eggs_data_dict = dict(zip(names, prices))

    df = pd.DataFrame(eggs_data_dict)
    for name in raw_eggs_data_dict:
        category = get_category(name)
        if category:
            new_row = pd.DataFrame({"Название" : [name], "Категория" : [category], "Цена" : [raw_eggs_data_dict[name]]})
            df = pd.concat([df, new_row], ignore_index = True)
            
    df['Категория'] = pd.Categorical(df['Категория'], categories=categories, ordered=True)
    df['Количество яиц в упаковке'] = df['Название'].str.extract(r'(\d+)шт')
    df['Количество яиц в упаковке'] = pd.to_numeric(df['Количество яиц в упаковке'])
    df['Цена одного яйца'] = df['Цена'] / df['Количество яиц в упаковке']
    df = df.sort_values('Категория')
    df = df.reset_index(drop = True)
    df['Магазин'] = shop
    return df

In [ ]:
df1 = get_shop_egg_prices("perek")
df2 = get_shop_egg_prices("pyaterka")

In [ ]:
df = pd.concat([df1, df2], ignore_index = True)
df = df.sort_values('Категория')
df = df.reset_index(drop = True)
df